# 26 · A fair contest on the 3,993 outside matches

> **Run on the first data file.** This research notebook and its saved outputs come from the ball-by-ball file as first supplied (every extra counted as 1 run). On 16 Nov 2026 the organisers issued a corrected file with the real extra runs; the method was re-run and re-checked on it in [notebook 28](28_final_review_corrected_data.ipynb), and both top-10 lists kept the same names.

Every test so far kept the current ranking. One honest reason: its settings were tuned on the same 52 internal checks used to judge the challengers, which gives it a head start there. The outside matches were never used to tune anything, so here every version starts equal: the current rating, the position, job and style versions from notebooks 24-25, raw pressure strike rate, and overall level without pressure. The rule is symmetric and was fixed in advance.

**Result.** Over all outside matches it is a **tie**: current 0.120, same style 0.122, raw pressure strike rate 0.122 (no version beats another with any confidence). The position versions are clearly worse. The current rating is the most consistent: never poor in any kind of cricket, and the best or close to it in World Cups and full-member T20Is, where the style and strike-rate versions do badly. In franchise leagues those two look better, but that check was added after seeing the results and could be chance. Verdict: the current ranking is **not beaten**, not proven best.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes and the rule fixed before running</summary>

P29 - A fair contest for the batting ranking on the 3,993 outside matches (research; nothing existing is changed).  
  
Why: the user pointed out that every test so far kept the current ranking. One real reason: the current rating's settings  
were tuned on the same 52 internal checks used to judge the challengers, which gives it a head start there. The outside  
(Cricsheet) matches were never used to tune anything, so every version starts equal.  
Versions (all built on the full supplied data with the same settings):  
  current; same batting position (P27 A); same position + phase (P27 B); same role (P28); same style (P28);  
  same role + style (P28); raw pressure strike rate; overall level without pressure.  
Yardstick: exactly P14's (pressure balls in public matches, players with 20+ such balls, available players only):  
  rank correlation with win chance added per pressure ball, and with runs added per pressure ball.  
Rule fixed BEFORE running (symmetric: no favour to the current ranking):  
  outside score = average of the two rank correlations on ALL public matches.  
  1,000 paired bootstraps over outside matches give P(version beats current).  
  P >= 0.90 -> the version is better: replace the current batting ranking with it.  
  0.70 <= P < 0.90 -> possibly better: report it as a real warning.  
  otherwise -> no evidence it is better.  
Sequential, 2 threads, low priority.
</details>

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from rrlib.common import load, load_json, load_table, save_table, save_json, get_logger, Timer, weighted_spearman, excluded_players
from rrlib import ratings as RL
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

log = get_logger("p29_fair_contest")
pd.set_option("display.width", 220)
v = load("values")
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]["bat"]
v["w_"] = v.pressure_pct.clip(1e-3, 1) ** 2
fb = v.faced.astype(bool) & v.v3_bat.notna()
W = v.w_ * v.v3_bat

# ---- P27: batting position (1-2, 3-4, 5-6, 7+) and position x phase
v["slot"] = pd.cut(v.batting_order, [0, 2, 4, 6, 11], labels=False).fillna(3).astype(int).values


def rel(keys, name):
    g = pd.DataFrame({"wv": W[fb], "w": v.w_[fb]}).join(v.loc[fb, keys])
    o = g.groupby(keys)[["wv", "w"]].sum()
    off = (o.wv / o.w)
    idx = pd.MultiIndex.from_frame(v[keys]) if len(keys) > 1 else v[keys[0]]
    v[name] = v.v3_bat - off.reindex(idx).fillna(0).values


rel(["slot"], "v9_bat"); rel(["slot", "phase"], "v10_bat")
# ---- P28: role (usage) and style (normal balls, against expected)
roles = load("roles"); ROLE = roles[roles.side == "bat"].set_index("player_id").role
v["role"] = v.striker_id.map(ROLE).fillna("Unassigned").values
b = v[fb & ~v.is_pressure.astype(bool)]
g = b.groupby("striker_id").agg(n=("r_bdry_bat", "size"), power=("r_bdry_bat", "mean"), dots=("r_dot_bat", "mean"))
st = np.select([(g.power >= 0) & (g.dots < 0), (g.power >= 0) & (g.dots >= 0), (g.power < 0) & (g.dots < 0)],
               ["Accelerator", "Hitter", "Anchor"], "Slower scorer")
STY = pd.Series(np.where(g.n >= 120, st, "Unclassified"), index=g.index)
v["sty"] = v.striker_id.map(STY).fillna("Unclassified").values
v["rs"] = v.role + " | " + v.sty
rel(["role"], "v11_bat"); rel(["sty"], "v12_bat")
n_rs = v[fb].groupby("rs").striker_id.nunique()
gg = pd.DataFrame({"wv": W[fb], "w": v.w_[fb], "rs": v.rs[fb]}).groupby("rs")[["wv", "w"]].sum()
o_rs = (gg.wv / gg.w)[n_rs.reindex(gg.index) >= 5]
o_role = pd.DataFrame({"wv": W[fb], "w": v.w_[fb], "role": v.role[fb]}).groupby("role")[["wv", "w"]].sum()
o_role = o_role.wv / o_role.w
v["v13_bat"] = v.v3_bat - v.rs.map(o_rs).fillna(v.role.map(o_role)).fillna(0).values

SC = {"current": load_table("final_all_bat").set_index("player_id").score}
LAB = {"v9": "same batting position", "v10": "same position + phase", "v11": "same role", "v12": "same style",
       "v13": "same role + style"}
with Timer(log, "full-data scores"):
    for met, lab in LAB.items():
        SC[lab] = RL.build_scores(v, "bat", dict(chosen, value=met), comp).score
    t = v[v.is_pressure & v.faced.astype(bool)]
    SC["raw pressure strike rate"] = t.groupby("striker_id").runs.mean()
    SC["overall level, no pressure"] = RL.build_scores(v, "bat", dict(chosen, model="R2-all", min_n=0)).score

# ---- outside yardstick (same as P14)
ve = load("values_ext")
players = load("players").set_index("player_id")
EXCL = excluded_players()
base_ids = set(v.loc[v.is_pressure & v.v1_bat.notna(), "striker_id"].unique())
GROUPS = {"all public matches": np.ones(len(ve), bool), "T20 World Cups": ve.is_wc.eq(1).values,
          "franchise leagues": ve.level.isin(["BBL", "PSL", "CPL", "SA20", "ILT20", "MLC", "BPL", "LPL", "Blast"]).values,
          "full-member T20Is (incl. World Cups)": ve.level.eq("T20I-full").values}


def evaluate(y):
    y = y[(y.n >= 20) & y.index.isin(players.index) & y.index.isin(base_ids) & ~y.index.isin(EXCL)]
    out = {}
    for name, sc in SC.items():
        ss = sc.reindex(y.index).fillna(-1e9)
        top = ss.sort_values(ascending=False).index[:10]
        out[name] = {"players": len(y), "rank_corr_winprob": weighted_spearman(ss.values, y.y.values, y.n.values),
                     "rank_corr_runs": weighted_spearman(ss.values, y.y1.values, y.n.values),
                     "top10_runs_per_100": 100 * (np.average(y.loc[top, "y1"], weights=y.loc[top, "n"]) - np.average(y.y1, weights=y.n)),
                     "top10_winprob_pts_per_100": 1e4 * (np.average(y.loc[top, "y"], weights=y.loc[top, "n"]) - np.average(y.y, weights=y.n))}
        out[name]["outside_score"] = 0.5 * (out[name]["rank_corr_winprob"] + out[name]["rank_corr_runs"])
    return out


rows = []
for gname, gm in GROUPS.items():
    d = ve[gm & ve.is_pressure & ve.wpa_bat.notna()]
    y = d.groupby("striker_id").agg(y=("wpa_bat", "mean"), y1=("v1_bat", "mean"), n=("wpa_bat", "size"))
    for name, r in evaluate(y).items():
        rows.append(dict(group=gname, version=name, **r))
R = pd.DataFrame(rows)
save_table(R, "research_fair_contest_outside")

# ---- paired bootstrap over outside matches (all public matches)
d = ve[ve.is_pressure & ve.wpa_bat.notna()]
mp = d.groupby(["match_id", "striker_id"]).agg(s=("wpa_bat", "sum"), s1=("v1_bat", "sum"), n=("wpa_bat", "size")).reset_index()
mids, mi = np.unique(mp.match_id.values, return_inverse=True)
pids, pi = np.unique(mp.striker_id.values, return_inverse=True)
rng = np.random.default_rng(2911)
B = 1000
boot = {k: [] for k in SC}
with Timer(log, f"{B} paired bootstraps over {len(mids)} outside matches"):
    for _ in range(B):
        c = np.bincount(rng.integers(0, len(mids), len(mids)), minlength=len(mids))[mi]
        n = np.bincount(pi, weights=c * mp.n.values, minlength=len(pids))
        ok = n > 0
        y = pd.DataFrame({"y": np.bincount(pi, weights=c * mp.s.values, minlength=len(pids))[ok] / n[ok],
                          "y1": np.bincount(pi, weights=c * mp.s1.values, minlength=len(pids))[ok] / n[ok], "n": n[ok]}, index=pids[ok])
        for name, r in evaluate(y).items():
            boot[name].append(r["outside_score"])
Bt = pd.DataFrame(boot)
summ = {}
allg = R[R.group == "all public matches"].set_index("version")
for name in SC:
    dd = Bt[name] - Bt["current"]
    p = float((dd > 0).mean())
    summ[name] = {"outside_score": float(allg.loc[name, "outside_score"]), "winprob": float(allg.loc[name, "rank_corr_winprob"]),
                  "runs": float(allg.loc[name, "rank_corr_runs"]), "diff_vs_current": float(dd.mean()),
                  "ci90": [float(dd.quantile(0.05)), float(dd.quantile(0.95))], "P_beats_current": p,
                  "verdict": ("current" if name == "current" else "BETTER: replace" if p >= 0.9 else
                              "possibly better: warning" if p >= 0.7 else "no evidence it is better")}
save_json(summ, "research_fair_contest_summary")
log.info("\nALL PUBLIC MATCHES (pre-registered decision):\n" + pd.DataFrame(summ).T.round(3).to_string())
log.info("\nBy group (outside score = average of the two rank correlations):\n" +
         R.pivot(index="version", columns="group", values="outside_score").round(3).to_string())
log.info("\nTop-10 runs per 100 pressure balls, by group:\n" +
         R.pivot(index="version", columns="group", values="top10_runs_per_100").round(1).to_string())


# ---- POST HOC (added after seeing the results above; not part of the pre-registered decision):
# the same paired bootstrap inside each kind of competition. With 7 versions x 3 groups = 21 comparisons,
# about 2 readings of P >= 0.90 are expected by chance alone.
rows = []
for gname, gm in GROUPS.items():
    if gname == "all public matches":
        continue
    d = ve[gm & ve.is_pressure & ve.wpa_bat.notna()]
    mp = d.groupby(["match_id", "striker_id"]).agg(s=("wpa_bat", "sum"), s1=("v1_bat", "sum"), n=("wpa_bat", "size")).reset_index()
    mids, mi = np.unique(mp.match_id.values, return_inverse=True)
    pids, pi = np.unique(mp.striker_id.values, return_inverse=True)
    rng = np.random.default_rng(2912)
    boot = {k: [] for k in SC}
    for _ in range(1000):
        c = np.bincount(rng.integers(0, len(mids), len(mids)), minlength=len(mids))[mi]
        n = np.bincount(pi, weights=c * mp.n.values, minlength=len(pids))
        ok = n > 0
        y = pd.DataFrame({"y": np.bincount(pi, weights=c * mp.s.values, minlength=len(pids))[ok] / n[ok],
                          "y1": np.bincount(pi, weights=c * mp.s1.values, minlength=len(pids))[ok] / n[ok], "n": n[ok]}, index=pids[ok])
        for k, r in evaluate(y).items():
            boot[k].append(r["outside_score"])
    Bt = pd.DataFrame(boot)
    for k in SC:
        if k == "current":
            continue
        dd = Bt[k] - Bt["current"]
        rows.append({"group": gname, "version": k, "diff": dd.mean(), "lo90": dd.quantile(.05), "hi90": dd.quantile(.95),
                     "P_beats_current": (dd > 0).mean()})
T = pd.DataFrame(rows)
save_table(T, "research_fair_contest_bygroup_posthoc")
log.info("
POST HOC, by kind of competition (P that the version beats the current ranking):
" +
         T.pivot(index="version", columns="group", values="P_beats_current").round(2).to_string())
